# 2주차 실습 (07) - 베이즈 정리 & 나이브 베이즈 from scratch

베이즈 정리로 사후확률을 계산하고, **라플라스 스무딩 + log 공간**으로
나이브 베이즈 스팸 분류기를 만든다. Beta 켤레 사전으로 순차 갱신도 확인한다.

## Step 1. 베이즈 정리 — 기저율 오류

`P(H|E) = P(E|H)P(H) / [P(E|H)P(H) + P(E|¬H)P(¬H)]`. 99% 정확한 검사도 희귀병이면 양성 시 실제 병일 확률이 1% 미만.

In [ ]:
def bayes(prior, likelihood, false_positive_rate):
    evidence = likelihood * prior + false_positive_rate * (1 - prior)
    return likelihood * prior / evidence


p1 = bayes(prior=1e-4, likelihood=0.99, false_positive_rate=0.01)
print(f"1회 양성:  P(sick|+) = {p1:.4f}  ({p1 * 100:.2f}%)")

# 사후를 사전으로 재사용 -> 두 번 양성
p2 = bayes(prior=p1, likelihood=0.99, false_positive_rate=0.01)
print(f"2회 양성:  P(sick|+,+) = {p2:.4f}  ({p2 * 100:.2f}%)")

## Step 2. 나이브 베이즈 분류기

특징(단어) 조건부 독립 가정 + 라플라스 스무딩. log 공간에서 더해 언더플로 방지.

In [ ]:
import math
from collections import defaultdict


class NaiveBayes:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.class_counts = defaultdict(int)
        self.word_counts = defaultdict(lambda: defaultdict(int))
        self.class_word_totals = defaultdict(int)
        self.vocab = set()

    def train(self, documents, labels):
        for doc, label in zip(documents, labels):
            self.class_counts[label] += 1
            for word in doc.lower().split():
                self.word_counts[label][word] += 1
                self.class_word_totals[label] += 1
                self.vocab.add(word)

    def predict(self, document):
        words = document.lower().split()
        total_docs = sum(self.class_counts.values())
        vocab_size = len(self.vocab)
        best_class, best_score = None, float("-inf")
        for cls in self.class_counts:
            score = math.log(self.class_counts[cls] / total_docs)   # log 사전
            for word in words:
                count = self.word_counts[cls].get(word, 0)
                total = self.class_word_totals[cls]
                score += math.log((count + self.smoothing)
                                  / (total + self.smoothing * vocab_size))
            if score > best_score:
                best_class, best_score = cls, score
        return best_class

## Step 3. 스팸 데이터로 학습·예측

In [ ]:
docs = [
    "win free money now", "free lottery ticket winner",
    "claim your prize today free", "urgent offer free cash",
    "congratulations you won free",
    "meeting tomorrow at noon", "project update attached",
    "can we schedule a call", "quarterly report review",
    "lunch on thursday sounds good", "team standup notes attached",
    "please review the pull request",
]
labels = ["spam"] * 5 + ["ham"] * 7

clf = NaiveBayes()
clf.train(docs, labels)

for msg in ["free money waiting for you", "meeting rescheduled to friday",
            "you won a free prize", "please review the attached report"]:
    print(f"  '{msg}' -> {clf.predict(msg)}")

## Step 4. 순차 베이즈 갱신 (Beta 켤레 사전)

`Beta(a,b) + s성공/f실패 → Beta(a+s, b+f)`. 오늘의 사후 = 내일의 사전. 적분 없이 덧셈만.

In [ ]:
a, b = 1, 1                              # Beta(1,1) = 균등 사전
print(f"사전    Beta({a},{b})  mean={a / (a + b):.3f}")
for heads, tails in [(7, 3), (5, 5)]:
    a, b = a + heads, b + tails          # 사후 = Beta(a+성공, b+실패)
    print(f"+{heads}H/{tails}T -> Beta({a},{b})  mean={a / (a + b):.3f}")

## 정리

- 사전 = 정칙화(ridge = 가우시안 사전), 사후 = 불확실성, 베이즈 갱신 = 온라인 학습.
- 나이브 베이즈: 독립 가정은 틀려도 순위만 맞으면 잘 작동.
- 개념 노트: [[🧮 베이즈 정리와 나이브 베이즈]] (Obsidian LLM_Wiki)